# Module 7 — Production, Safety and Observability

**Course:** Build Your First AI Agent from Scratch  
**Community:** [Autonomous MSP](https://skool.com/autonomous-msp-2162)

**What you build in this module:**
- Enhanced approval gate with full audit logging
- LangSmith tracing (one environment variable — that's all)
- Prometheus metrics for dashboards
- Two-credential RBAC model (read-only vs write)
- Infinite loop protection (iteration cap + wall time)
- Production readiness checklist

**Rule:** An agent that operators don't trust will not be used. An agent they can observe, audit, and control will be adopted and become valuable over time.

## Step 1 — Install Dependencies

In [ ]:
!pip install langsmith prometheus_client python-dotenv -q

## Part 1 — The Enhanced Approval Gate

This builds on the Module 3 safety gate. The new version:
- Writes a structured JSON audit log for every action
- Shows config diffs before WRITE approvals
- Requires `YES I CONFIRM` for ADMIN operations
- Logs rejections — not just approvals

**The audit log is a client deliverable.** When a client's auditor asks "did a human approve every change?" — you hand them this file.

In [ ]:
import logging
import datetime
import json
import time

AUDIT_LOG_PATH = "agent_audit.log"

# Dedicated audit logger — separate from application logs
audit_logger = logging.getLogger("agent.audit")
audit_logger.setLevel(logging.INFO)

file_handler = logging.FileHandler(AUDIT_LOG_PATH)
file_handler.setFormatter(logging.Formatter("%(message)s"))  # raw JSON, no extra formatting
audit_logger.addHandler(file_handler)


def log_tool_execution(
    tool_name: str,
    params:    dict,
    approved:  bool,
    operator:  str = "",
    duration_ms: float = 0.0
):
    """
    Write one structured JSON record to the audit log.
    Called for EVERY tool execution — approved, rejected, or auto-executed.
    """
    audit_logger.info(json.dumps({
        "tool":        tool_name,
        "params":      params,
        "approved":    approved,
        "operator":    operator or "auto",
        "duration_ms": round(duration_ms, 2),
        "timestamp":   datetime.datetime.utcnow().isoformat() + "Z",
    }))


# Test it
log_tool_execution("run_show_command", {"device": "rtr-01", "command": "show ip ospf"}, approved=True)
print("Wrote one audit entry. Reading it back:")
with open(AUDIT_LOG_PATH) as f:
    last_line = f.readlines()[-1]
    print(json.dumps(json.loads(last_line), indent=2))

In [ ]:
# Copy ToolCategory / ToolResult / BaseTool from Module 3
# (or import from your tools module in a real project)
from enum import Enum
from dataclasses import dataclass
from abc import ABC, abstractmethod

class ToolCategory(Enum):
    READ  = "read"
    WRITE = "write"
    ADMIN = "admin"

@dataclass
class ToolResult:
    success: bool
    data: dict
    error: str = ""
    tool_name: str = ""
    execution_time_ms: float = 0.0

class BaseTool(ABC):
    name: str = ""
    description: str = ""
    category: ToolCategory = ToolCategory.READ
    parameters: dict = {}

    @abstractmethod
    def execute(self, **kwargs) -> ToolResult:
        pass


def execute_with_approval(
    tool:              BaseTool,
    params:            dict,
    auto_approve_reads: bool = True,
    operator:          str  = "unknown",
) -> ToolResult:
    """
    Production-grade approval gate.
    
    READ  → auto-execute, log it
    WRITE → show tool + params + diff, require y/n, log decision
    ADMIN → require 'YES I CONFIRM', log decision
    """

    # ── READ: auto-execute ────────────────────────────────────
    if tool.category == ToolCategory.READ and auto_approve_reads:
        start  = time.time()
        result = tool.execute(**params)
        result.execution_time_ms = (time.time() - start) * 1000
        result.tool_name = tool.name
        log_tool_execution(tool.name, params, approved=True,
                           operator=operator, duration_ms=result.execution_time_ms)
        return result

    # ── WRITE: require y/n ────────────────────────────────────
    if tool.category == ToolCategory.WRITE:
        print("\n" + "=" * 60)
        print("  APPROVAL REQUIRED")
        print("=" * 60)
        print(f"  Tool    : {tool.name}")
        print(f"  Action  : {tool.description}")
        print(f"  Params  : {json.dumps(params, indent=4)}")

        # Show config diff if the tool supports it
        if hasattr(tool, "preview_change"):
            preview = tool.preview_change(**params)
            if preview:
                print(f"\n  DIFF:\n{preview}")

        print("=" * 60)
        operator_name = input("  Your name: ").strip()
        decision      = input("  Approve? (y/n): ").strip().lower()
        approved      = (decision == "y")

        log_tool_execution(tool.name, params, approved=approved, operator=operator_name)

        if not approved:
            return ToolResult(success=False, data={},
                              error=f"Rejected by {operator_name}",
                              tool_name=tool.name)

    # ── ADMIN: require explicit phrase ────────────────────────
    elif tool.category == ToolCategory.ADMIN:
        print("\n" + "!" * 60)
        print("  ADMIN OPERATION — HIGH RISK")
        print(f"  Tool: {tool.name}")
        print(f"  Params: {params}")
        confirm  = input("  Type 'YES I CONFIRM' to proceed: ").strip()
        approved = (confirm == "YES I CONFIRM")
        log_tool_execution(tool.name, params, approved=approved, operator="manual-confirm")
        if not approved:
            return ToolResult(success=False, data={},
                              error="Admin confirmation failed", tool_name=tool.name)

    start  = time.time()
    try:
        result = tool.execute(**params)
        result.execution_time_ms = (time.time() - start) * 1000
        result.tool_name = tool.name
        return result
    except Exception as e:
        return ToolResult(success=False, data={}, error=str(e), tool_name=tool.name,
                          execution_time_ms=(time.time() - start) * 1000)


print("Enhanced approval gate ready.")

## Part 2 — LangSmith Tracing

LangSmith gives you a complete trace of every agent run in a web UI: which nodes ran, how long each took, what the LLM was thinking, what tools fired.

**Setup cost:** 3 environment variables. That's it. No code changes to your agent.

In [ ]:
import os

# ── LangSmith setup — add these BEFORE any LangChain/LangGraph imports ──
# Get your key at smith.langchain.com

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"]    = "network-ops-agent"  # project name in LangSmith UI
os.environ["LANGCHAIN_API_KEY"]    = os.environ.get("LANGSMITH_API_KEY", "ls__your_key_here")

print("LangSmith tracing configured.")
print("Every agent.invoke() call now sends a trace to smith.langchain.com")
print()
print("What you will see in LangSmith:")
print("""
Run: "OSPF neighbor INIT state on rtr-01"
├─ observe (2.1s)
│   └─ ChromaDB query: "OSPF INIT" → 2 results
├─ reason (3.2s)
│   └─ LLM call → TOOL_CALL: run_show_command
├─ act (1.8s)
│   └─ Tool: run_show_command(rtr-01, "show ip ospf neighbor")
├─ reason (2.9s)
│   └─ LLM call → TOOL_CALL: check_ospf_config
├─ act (1.2s)
│   └─ Tool: check_ospf_config(rtr-01, Gi0/1) → area=1 (mismatch)
├─ reason (2.1s)
│   └─ LLM call → CONCLUDE (confidence=0.92)
└─ verify (3.8s)
    └─ LLM call → FINAL_ANSWER
""")

## Part 3 — Prometheus Metrics

LangSmith gives you reasoning traces. Prometheus gives you operational metrics: query volume, latency, confidence distribution, tool success rates. These go into Grafana dashboards.

Key metric to watch: **agent_last_confidence**. If it consistently stays below 0.5, your agent is guessing — your knowledge base needs more data.

In [ ]:
from prometheus_client import Counter, Histogram, Gauge, start_http_server

# ── Metric definitions ─────────────────────────────────────────
QUERIES_TOTAL = Counter(
    "agent_queries_total",
    "Total queries processed",
    ["status"],          # label: success / failed
)

QUERY_LATENCY = Histogram(
    "agent_query_latency_seconds",
    "Time from query to final answer",
    buckets=[5, 10, 30, 60, 120, 300],  # seconds
)

TOOL_CALLS = Counter(
    "agent_tool_calls_total",
    "Tool invocations by name and outcome",
    ["tool_name", "outcome"],  # labels: tool_name, outcome (success/failed/rejected)
)

CONFIDENCE_GAUGE = Gauge(
    "agent_last_confidence",
    "Confidence score of the last completed query",
)

ITERATIONS_HISTOGRAM = Histogram(
    "agent_iterations_per_query",
    "Number of OBSERVE-REASON-ACT cycles per query",
    buckets=[1, 2, 3, 5, 8, 13],
)


def record_query_result(state: dict, start_time: float):
    """
    Call this after agent.invoke() completes.
    Records all metrics for the completed query.
    """
    duration = time.time() - start_time
    status   = "success" if state.get("done") else "failed"

    QUERIES_TOTAL.labels(status=status).inc()
    QUERY_LATENCY.observe(duration)
    CONFIDENCE_GAUGE.set(state.get("confidence", 0))
    ITERATIONS_HISTOGRAM.observe(state.get("iteration", 0))

    for tc in state.get("tool_calls", []):
        # tc can be a string (tool name) or dict with success/tool_name
        if isinstance(tc, dict):
            outcome   = "success" if tc.get("success") else "failed"
            tool_name = tc.get("tool_name", "unknown")
        else:
            outcome   = "success"
            tool_name = tc
        TOOL_CALLS.labels(tool_name=tool_name, outcome=outcome).inc()


def start_metrics_server(port: int = 9090):
    """Expose Prometheus metrics at /metrics on the given port."""
    start_http_server(port)
    print(f"Metrics available at http://localhost:{port}/metrics")


# Demo: record a fake query result
fake_state = {
    "done":       True,
    "confidence": 0.92,
    "iteration":  3,
    "tool_calls": ["show_ospf_neighbors", "show_ospf_interface"],
}
start = time.time() - 45  # pretend it took 45 seconds
record_query_result(fake_state, start)

print("Metrics recorded. Current confidence gauge:", CONFIDENCE_GAUGE._value.get())

## Part 4 — Credentials and RBAC

Two separate SSH accounts:
- **Read-only account** — for all `show` commands. No config privilege.
- **Write account** — only loaded after human approval. Higher privilege.

If the agent is ever compromised, the read-only account can't change anything.

In [ ]:
# ── config.py pattern ─────────────────────────────────────────
# In production: load from Vault or AWS Secrets Manager
# Here: load from environment variables

from dotenv import load_dotenv
try:
    load_dotenv()  # reads .env file if present
except Exception:
    pass  # no .env file found — env vars loaded from system


class Config:
    # Hard fail if missing — never silently use an empty API key
    ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")

    # Read-only SSH account — used for all show commands
    # On Cisco: username netops-agent-ro privilege 1
    READ_CREDENTIALS = {
        "username": os.environ.get("DEVICE_SSH_USER_READONLY", "netops-ro"),
        "password": os.environ.get("DEVICE_SSH_PASS_READONLY", ""),
    }

    # Write SSH account — only loaded after human approval in execute_with_approval()
    # On Cisco: username netops-agent-rw privilege 15
    WRITE_CREDENTIALS = {
        "username": os.environ.get("DEVICE_SSH_USER_WRITE", "netops-rw"),
        "password": os.environ.get("DEVICE_SSH_PASS_WRITE", ""),
    }

    MAX_ITERATIONS     = int(os.environ.get("AGENT_MAX_ITERATIONS", "10"))
    MAX_WALL_TIME_SECS = int(os.environ.get("AGENT_MAX_WALL_TIME",  "300"))


print("Config loaded.")
print(f"  Read user:  {Config.READ_CREDENTIALS['username']}")
print(f"  Write user: {Config.WRITE_CREDENTIALS['username']}")
print(f"  Max iterations: {Config.MAX_ITERATIONS}")
print(f"  Max wall time:  {Config.MAX_WALL_TIME_SECS}s")
print()
print("IOS device config to match:")
print("  username netops-agent-ro privilege 1 secret [strong-password]")
print("  username netops-agent-rw privilege 15 secret [different-strong-password]")

## Part 5 — Infinite Loop Protection

Two safeguards work together:
1. **Max iterations** — hard ceiling on how many reason-act cycles can run
2. **Wall time limit** — if the agent takes more than 5 minutes, force it to verify with what it has

Add both to your routing function.

In [ ]:
def route_after_reason_safe(state: dict) -> str:
    """
    Production routing function with all safety checks.
    This is the Module 5 routing function + wall time protection.
    """
    # Wall time check — force verify if we've been running too long
    elapsed = time.time() - state.get("start_time", time.time())
    if elapsed > state.get("max_wall_time_seconds", 300):
        print(f"  [route] Wall time limit reached ({elapsed:.0f}s) → force verify")
        return "verify"

    if state.get("done"):
        return "verify"

    if state.get("confidence", 0) >= 0.85:
        return "verify"

    if state.get("iteration", 0) >= state.get("max_iterations", 10):
        print(f"  [route] Max iterations reached → force verify")
        return "verify"

    if state.get("pending_tool") == "DONE":
        return "verify"

    return "act"


# Test the wall time protection
test_state = {
    "done":                False,
    "confidence":          0.4,
    "iteration":           3,
    "max_iterations":      10,
    "pending_tool":        "some_tool",
    "start_time":          time.time() - 310,   # pretend it started 310s ago
    "max_wall_time_seconds": 300,
}
print("Route with expired wall time:", route_after_reason_safe(test_state))

test_state["start_time"] = time.time()   # reset to now
print("Route with fresh start time:",  route_after_reason_safe(test_state))

## Production Readiness Checklist

Before pointing your agent at a real network, go through every item below.

In [ ]:
CHECKLIST = {
    "Safety": [
        "All WRITE tools require human approval (tested by pressing 'n')",
        "Agent uses read-only SSH credentials for show commands",
        "Separate write credentials only loaded after approval",
        "Config diff shown before every config push",
        "Audit log enabled and writing to file",
        "Max iterations set (default 10)",
        "Wall time limit set (default 300 seconds)",
    ],
    "Security": [
        "No credentials in code — all from environment variables",
        ".env file in .gitignore",
        "LangSmith API key not logged or printed",
        "Audit log stored on secure system",
    ],
    "Observability": [
        "LangSmith tracing enabled",
        "Prometheus metrics endpoint running",
        "Alert if confidence consistently below 0.5 (agent is guessing)",
    ],
    "Reliability": [
        "Agent tested with demo mode before real devices",
        "Tested on lab devices before production",
        "Fallback: if agent fails, operations continue manually",
        "Rollback plan documented",
    ],
    "Governance": [
        "Operators trained on approving/rejecting agent proposals",
        "Clear escalation path if agent suggests something unexpected",
        "Regular review of audit logs (weekly minimum)",
    ],
}

print("PRODUCTION READINESS CHECKLIST")
print("=" * 50)
for category, items in CHECKLIST.items():
    print(f"\n{category}:")
    for item in items:
        print(f"  [ ] {item}")

## What's Next

You have all the pieces. **Module 8** wires everything together into one complete, runnable OSPF troubleshooting agent — the capstone project.

---

**Module Challenge:** Run a few agent queries and inspect the audit log. Can you answer: (1) what query was run, (2) which tools were called, (3) were any WRITE tools approved or rejected? If any question can't be answered from the log, your audit logging is incomplete.